# OpenCV LBPH 얼굴인식 — 실제 실습 12단계
Google Colab에서 위에서 아래로 실행하세요. 얼굴 검출은 Haar Cascade, 얼굴 인식은 OpenCV LBPH를 사용합니다. PyTorch, FaceNet은 사용하지 않습니다.
주인 사진은 faces.zip 안의 owner/01.jpg ~ 08.jpg 형식입니다. 모델 파일 my_face.yml과 얼굴 사진은 공개 GitHub에 업로드하지 마세요.


## 01. OpenCV 설치


In [ ]:
%pip install -q opencv-contrib-python

## 02. OpenCV와 LBPH 기능 확인


In [ ]:
import cv2
import numpy as np
from pathlib import Path
print("OpenCV:", cv2.__version__)
assert hasattr(cv2, "face") and hasattr(cv2.face, "LBPHFaceRecognizer_create"), "opencv-contrib-python 설치를 확인하세요."
print("LBPH 사용 가능")

## 03. faces.zip 업로드


In [ ]:
from google.colab import files
uploaded = files.upload()
assert "faces.zip" in uploaded, "owner/01.jpg ~ 08.jpg를 담은 faces.zip을 업로드하세요."

## 04. ZIP 압축 해제 및 owner 사진 확인


In [ ]:
import zipfile
with zipfile.ZipFile("faces.zip") as z:
    for info in z.infolist():
        p = Path(info.filename)
        if p.is_absolute() or ".." in p.parts:
            raise ValueError("잘못된 ZIP 경로")
    z.extractall("data")
owner_dir = Path("data/owner")
if not owner_dir.exists():
    matches = list(Path("data").rglob("owner"))
    assert matches, "ZIP 내부에 owner 폴더가 없습니다."
    owner_dir = matches[0]
photos = sorted(p for p in owner_dir.iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png"})
print("사진 개수:", len(photos))
print([p.name for p in photos])
assert photos, "owner 폴더에 사진이 없습니다."

## 05. Haar Cascade 얼굴 검출기 준비


In [ ]:
cascade_path = cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
face_detector = cv2.CascadeClassifier(cascade_path)
assert not face_detector.empty(), "Haar Cascade 로드 실패"
print("Haar Cascade 준비 완료")

## 06. 얼굴 검출 및 200×200 흑백 전처리


In [ ]:
face_images = []
for path in photos:
    image = cv2.imread(str(path))
    if image is None:
        print("이미지 읽기 실패:", path.name)
        continue
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    boxes = face_detector.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(40, 40))
    if len(boxes) == 0:
        print("얼굴 검출 실패:", path.name)
        continue
    x, y, w, h = max(boxes, key=lambda b: b[2] * b[3])
    face = cv2.resize(gray[y:y+h, x:x+w], (200, 200))
    face_images.append(face)
    print("검출 완료:", path.name)
assert face_images, "학습할 얼굴이 없습니다."
print("학습 얼굴:", len(face_images))

## 07. LBPH 얼굴인식 모델 생성


In [ ]:
recognizer = cv2.face.LBPHFaceRecognizer_create()
print("LBPH 모델 생성 완료")

## 08. 주인 얼굴 학습


In [ ]:
labels = np.full(len(face_images), 1, dtype=np.int32)
recognizer.train(face_images, labels)
print("학습 완료: label=1 (OWNER)")

## 09. my_face.yml 저장


In [ ]:
recognizer.write("my_face.yml")
print("저장 완료: my_face.yml")

## 10. test.jpg 업로드 및 전처리


In [ ]:
test_upload = files.upload()
assert test_upload, "test.jpg를 업로드하세요."
test_name = next(iter(test_upload))
test_image = cv2.imread(test_name)
assert test_image is not None, "이미지 파일을 읽을 수 없습니다."
test_gray = cv2.cvtColor(test_image, cv2.COLOR_BGR2GRAY)
test_boxes = face_detector.detectMultiScale(test_gray, 1.1, 5, minSize=(40,40))
assert len(test_boxes), "테스트 사진에서 얼굴을 검출하지 못했습니다."
x, y, w, h = max(test_boxes, key=lambda b: b[2] * b[3])
test_face = cv2.resize(test_gray[y:y+h, x:x+w], (200, 200))
print("테스트 얼굴 전처리 완료")

## 11. LBPH 예측 및 거리 해석


In [ ]:
label, distance = recognizer.predict(test_face)
threshold = 60.0  # 수업용 임시 기준, 실제 데이터로 검증 필요
result = "OWNER" if label == 1 and distance < threshold else "UNKNOWN"
print(f"label={label}, distance={distance:.2f}")
print("판별 결과:", result)
print("주의: distance는 확률(%)이 아닙니다. 일반적으로 낮을수록 비슷합니다.")

## 12. 모델 다운로드 및 웹캠 실습 연결


In [ ]:
files.download("my_face.yml")
print("Windows PC에서 opencv-contrib-python 설치 후 다음 명령 실행:")
print("python src_opencv/webcam.py --model my_face.yml")
print("웹캠 창은 Q 키로 종료합니다.")